## 5-3. 量子近似最適化アルゴリズム（QAOA）

この節では、離散的な変数に対する**組み合わせ最適化問題**を解くためのアルゴリズムである量子近似最適化アルゴリズム（quantum approximate optimization algorithm, QAOA）を学ぶ。
この章で学んだ VQE・QCL と同様に、QAOA は変分量子回路を用いた量子・古典ハイブリッドアルゴリズムであり、NISQ デバイスでの実行可能性が考慮されている。
なお、この節では $n$ 桁の古典ビット・量子ビットを左から順に $1,2,\cdots,n$ と名付けていることに注意してほしい。

### 問題設定

QAOA は組み合わせ最適化問題の（近似）解を求めるためのアルゴリズムである。
特に、$z = z_1 z_2 \cdots z_n$ という $n$ 桁の**古典ビット**列 $z$ に関して（$z_1,\cdots,z_n$ は 0 か 1）、コスト関数 $C(z)$ を最小化する問題を考える。
このコスト関数は

$$
C(z)=\sum_{\alpha=1}^{M} C_\alpha(z), \qquad
C_\alpha(z)=T_\alpha z_{i^{(\alpha)}_1} z_{i^{(\alpha)}_2} \cdots z_{i^{(\alpha)}_{m_\alpha}}
$$

という形に展開できると仮定する。
ここで $C_\alpha(z)$ は $z_1,\cdots,z_n$ のうち $z_{i^{(\alpha)}_1}, z_{i^{(\alpha)}_2}, \cdots, z_{i^{(\alpha)}_{m_\alpha}}$ という $m_\alpha$ 個の積でかける単項であり、$T_\alpha$ は実数の係数、$M$ は $C(z)$ に含まれる項の総数である。
例えば、$n=2$ に対して $C(z)=0.3 z_1 - 4 z_2 + 1.4 z_1 z_2$ といったコスト関数が考えられる。

$C(z)$ を最小化するには、$n$ について指数関数的に多い $2^n$ 通りのビット列 $z$ の中から最適なものを見つけ出す必要がある。
一般的な $C(z)$ に対して最適な $z$ を探す問題は、計算量理論的には非常に難しいことが知られており、**量子コンピュータを用いても解けない**だろうと考えられている。
それでも、特定の種類の問題に関しては量子コンピュータ（QAOA や本節末のコラムで紹介する量子アニーリング）で効率的に解けるかもしれない、あるいは良い近似解が得られるかもしれないといった期待があり、量子コンピュータを活用する研究が行われている。

### QAOA の変分量子状態とコスト関数

QAOA では、$n$ 桁のビット列 $z$ の最小化問題を解くために、$n$ 個の**量子ビット**を用いる。
そして、整数 $p=1,2,\cdots$ によって量子回路の複雑さが決まる次の $n$ 量子ビット状態を考える。

$$
\ket{s}=\ket{+}^{\otimes n}=\frac{1}{2^{n/2}}\sum_{k=0}^{2^n-1}\ket{k},
$$

$$
\ket{\boldsymbol{\beta},\boldsymbol{\gamma}}_p = U_X(\beta^{(p)})U_C(\gamma^{(p)})\cdots U_X(\beta^{(1)})U_C(\gamma^{(1)})\ket{s}.
$$

ここで $\ket{+}=\frac{1}{\sqrt{2}}(\ket{0}+\ket{1})$ は $X$ 演算子の固有値 1 の固有状態であり、$\ket{k}$ は整数 $k=0,1,\cdots,2^n-1$ の二進数表示に対応する状態である。
$\boldsymbol{\beta}=(\beta^{(1)},\cdots,\beta^{(p)})$、$\boldsymbol{\gamma}=(\gamma^{(1)},\cdots,\gamma^{(p)})$ は合計 $2p$ 個の変分パラメータである。
また、量子回路 $U_C(\gamma), U_X(\beta)$ は次のように定義される。

$$
U_C(\gamma)=e^{-i\gamma C(Z)}=\prod_\alpha e^{-i\gamma C_\alpha(Z)},
$$

$$
U_X(\beta)=e^{-i\beta \sum_{j=1}^{n} X_j}=\prod_{j=1}^{n} e^{-i\beta X_j},
$$

$$
C(Z)=\sum_{\alpha=1}^{M} C_\alpha(Z), \qquad
C_\alpha(Z)=T_\alpha \frac{1-Z_{i^{(\alpha)}_1}}{2} \frac{1-Z_{i^{(\alpha)}_2}}{2} \cdots \frac{1-Z_{i^{(\alpha)}_{m_\alpha}}}{2}.
$$

つまり、$C(Z)$ はコスト関数 $C(z)$ のビット列 $z_1,\cdots,z_n$ を各量子ビット上の $Z$ 演算子を用いた $(1-Z_1)/2,\cdots,(1-Z_n)/2$ に置き換えたもの（演算子）である。
$U_X(\beta)$ はもう少し単純で、各量子ビット上の $X$ 演算子によるパウリ回転ゲートの積である。

この状態 $\ket{\boldsymbol{\beta},\boldsymbol{\gamma}}_p$ は量子アニーリングという技術に由来しており、その意味を理解するには量子アニーリングに関する知識が必要になる。とりあえず、QAOA を使うだけならこういうものだと受け入れて使ってしまえばよい。量子アニーリングについては、本節末のコラムにまとめたので興味がある人は読んでみてほしい。

QAOA では、変分量子状態 $\ket{\boldsymbol{\beta},\boldsymbol{\gamma}}_p$ を用いた次のようなコスト関数を考える。

$$
F(\boldsymbol{\beta}, \boldsymbol{\gamma}) = {}_p \! \mel{\boldsymbol{\beta}, \boldsymbol{\gamma}}{C(Z)}{\boldsymbol{\beta}, \boldsymbol{\gamma}}_p.
$$

パラメータ $\boldsymbol{\beta},\boldsymbol{\gamma}$ を調整することでこのコスト関数を最小化し、その結果からもともとの最適化問題の解を探そうとするのが QAOA の骨子である。なぜこのコスト関数の最小化でもともとの最適化問題の解が得られるか説明しよう。注意すべき点は、$C(Z)$ がすべて $Z$ 演算子で構成されているため、古典ビット列 $k$ に対応する量子状態 $\ket{k}$ に対して $C(Z)\ket{k}=C(k)\ket{k}$ が成り立ち、コスト関数の値が単なる係数として現れることである。
これにより、$\ket{\boldsymbol{\beta},\boldsymbol{\gamma}}_p = \sum_{k=0}^{2^n-1} a_k \ket{k}$ と何らかの係数 $a_k$ で展開したとき、コスト関数は

$$
F(\boldsymbol{\beta}, \boldsymbol{\gamma}) = \sum_{k=0}^{2^n-1} \sum_{k'=0}^{2^n-1} a_{k'}^* a_k C(k) \braket{k'}{k} = \sum_{k=0}^{2^n-1} |a_k|^2 C(k)
$$

と、$k$ に対するコスト関数の値 $C(k)$ の重み付きの和になる。量子状態の規格化条件から $\sum_{k=0}^{2^n-1} |a_k|^2 = 1$ だから、コスト関数の値が最小になるのは、もともとの最適化問題のコスト $C(k)$ が最小となる $k^*$ のみに対して $a_{k^*}=1$ で他の $k$ については $a_k=0$ となるときである。言い換えると、コスト関数の最小は状態が $\ket{\boldsymbol{\beta},\boldsymbol{\gamma}}_p = \ket{k^*}$ となるときに実現される。
以上より、コスト関数 $F(\boldsymbol{\beta}, \boldsymbol{\gamma})$ の最小化を実現する状態 $\ket{\boldsymbol{\beta},\boldsymbol{\gamma}}_p$ があれば、その状態を測定することで、もともとの最適化問題のコスト $C(k)$ を最小化するようなビット列 $k^*$ が確率 1 で得られることになる。これが QAOA の要点である。

### QAOA の手順

最も単純な QAOA の手順は以下の通りである。

1. 量子コンピュータ上で重ね合わせ状態 $\ket{s}=\ket{+}^{\otimes n}$ を作る。
2. パラメータ $\boldsymbol{\beta},\boldsymbol{\gamma}$ に応じて、量子状態に $U_C(\gamma^{(i)}), U_X(\beta^{(i)})$（$i=1,\cdots,p$）をかけていき、状態 $\ket{\boldsymbol{\beta},\boldsymbol{\gamma}}_p$ を得る。
3. 量子コンピュータを用いて期待値 ${}_p\!\mel{\boldsymbol{\beta},\boldsymbol{\gamma}}{C(Z)}{\boldsymbol{\beta},\boldsymbol{\gamma}}_p$ を測定する。
4. 古典コンピュータで、上記の期待値がより小さくなるようにパラメータ $\boldsymbol{\beta},\boldsymbol{\gamma}$ を更新する。
5. ステップ 1 から 4 を繰り返し、最適なパラメータ $\boldsymbol{\beta}^*,\boldsymbol{\gamma}^*$ を得る。
6. 状態 $\ket{\boldsymbol{\beta},\boldsymbol{\gamma}}_p$ の測定を複数回実行し、得られた測定結果 $z$ に対して $C(z)$ を計算する。もっとも良かった $z$ を、元々の最適化問題の解として採用する。

ステップ 3 の期待値の測定は、一般の VQE の場合とは異なり、単に $\ket{\boldsymbol{\beta},\boldsymbol{\gamma}}_p$ を測定して得られた各ビット列 $k$ の確率振幅を用いて計算できる。
また、最後のステップ 6 では、$F(\boldsymbol{\beta},\boldsymbol{\gamma})$ の最小化が完璧であれば出現するビット列は常に元々の最適化問題の解であるが、実際には最小化が不完全であることも多いので、それをカバーするために「測定して得られたビット列のうち最も良いものを採用する」としてある。

### QAOA による量子アドバンテージの可能性

QAOA の実装に必要な 1 量子ビット・2 量子ビットのゲート数は、$U_C(\gamma)$ と $U_X(\beta)$ それぞれの実装に必要なゲート数を考慮すると、およそ $(O(M)+O(n))\cdot p$ である。
よって、$M$ も $p$ もそれほど大きくない場合は、QAOA は効率的に量子コンピュータで実行できる。
しかし、$p$ が $n$ の多項式程度の大きさのときに十分良い近似解が得られるような変分量子状態を作れるかは非自明であり、パラメータ $\boldsymbol{\beta},\boldsymbol{\gamma}$ がうまく最適化できる保証もない。
古典コンピュータの組み合わせ最適化問題用のソルバーも非常に強力であることと合わせて考えると、QAOA で量子アドバンテージを達成するのは簡単ではないことが予想される。
もちろん、達成が不可能であるという証明もないし、QAOA を改良したアルゴリズムも多数提案されているから、今後の研究の進展が期待される。

### 実装例：最大カット問題を QAOA で解く

QAOA の実装例として、最大カット問題を解いてみよう。
最大カット問題（max-cut problem）は、$n$ 個の頂点を持つグラフの頂点を 2 つのグループに分割するとき、分割される辺の数（つまり、グループ間をつなぐ辺の総数）の最大値を求める問題である。

[図 5.4 プレースホルダ: 最大カット問題の例]

この問題を QAOA で扱えるような最適化問題に帰着させるには、以下のようにする。
まず、$n$ 個の頂点を 2 つのグループに分けたとき、片方のグループに属する頂点に $\ket{0}$、もう一方のグループに属するものに $\ket{1}$ を割り当てる。
こうすることで、$2^n$ 通りの頂点の分け方と、$n$ 量子ビットの状態 $\ket{k}$ が対応する。
二つの頂点 $i,j$ に状態 $\ket{l},\ket{m}$ が割り当てられているとき、$l=m$ の場合は $Z_i Z_j \ket{l}\ket{m}=\ket{l}\ket{m}$、$l\neq m$ の場合は $Z_i Z_j \ket{l}\ket{m}=-\ket{l}\ket{m}$ が成り立つ。
よって、次の式で定義されるコスト関数は、「ビット列 $k$ に対応するグループ分けによって分割される辺の数 $\times (-1)$」に対応する。

$$
C(Z) = -\frac{1}{2}\sum_{\text{辺で繋がっている頂点 } i,j} (1-Z_i Z_j),
$$

$$
C(Z)\ket{k}=C(k)\ket{k} \qquad (k=0,1,\cdots,2^n-1).
$$

ゆえに、$C(k)$ を最小化するようなビット列 $k=k_1\cdots k_n$ を見つければ、分割する辺の数を最大化するような頂点の分け方を見つけたことになる。

#### 長方形の最大カット問題

[図 5.5 プレースホルダ: 4 つの辺と頂点を持つ長方形]

それでは、長方形の最大カット問題を解いてみよう。この場合、$C(Z)$ は

$$
C(Z)= -\frac{1}{2}(1-Z_0 Z_1)-\frac{1}{2}(1-Z_1 Z_2)-\frac{1}{2}(1-Z_2 Z_3)-\frac{1}{2}(1-Z_3 Z_0)
$$

すなわち

$$
C(Z)=\frac{1}{2}(Z_0 Z_1 + Z_1 Z_2 + Z_2 Z_3 + Z_3 Z_0) - 2
$$

となる。第二項は定数だから、ここからは

$$
C(Z)=\frac{1}{2}(Z_0 Z_1 + Z_1 Z_2 + Z_2 Z_3 + Z_3 Z_0)
$$

とおく。

まずは変分量子状態 $\ket{\boldsymbol{\beta},\boldsymbol{\gamma}}_p$ を実装しよう。
$U_C(\gamma)=\prod_{i=0}^{3} e^{-i\gamma Z_i Z_{i+1}}$ を実装するには、

$$
e^{-i\delta Z_i Z_{i+1}} = \operatorname{CNOT}_{i,i+1} \cdot e^{-i\delta Z_{i+1}} \cdot \operatorname{CNOT}_{i,i+1}
$$

を使う。
QURI Parts の変分量子回路を用いて $U_C(\gamma), U_X(\beta)$ を作成し、$\ket{\boldsymbol{\beta},\boldsymbol{\gamma}}_p$ を作成するコードは以下の通りである。
なお、QURI Parts の回転ゲートの定義は $U_Z(\theta)=e^{-i\theta Z/2}$ だから、パラメータを 2 倍して回転角に代入している。

```python
import numpy as np
from scipy.optimize import minimize
from quri_parts.circuit import LinearMappedUnboundParametricQuantumCircuit
from quri_parts.core.state import quantum_state
from quri_parts.core.operator import Operator, pauli_label
from quri_parts.qulacs.estimator import create_qulacs_vector_estimator
from quri_parts.qulacs.simulator import evaluate_state_to_vector

n = 4

def add_U_C(circuit, gamma_idx):
    gamma = circuit.add_parameter(f"gamma_{gamma_idx}")
    for i in range(n):
        j = (i + 1) % n
        circuit.add_CNOT_gate(i, j)
        circuit.add_ParametricRZ_gate(j, {gamma: 2})
        circuit.add_CNOT_gate(i, j)

def add_U_X(circuit, beta_idx):
    beta = circuit.add_parameter(f"beta_{beta_idx}")
    for i in range(n):
        circuit.add_ParametricRX_gate(i, {beta: 2})
    return circuit

def QAOA_state(x: list[float], p: int):
    circuit = LinearMappedUnboundParametricQuantumCircuit(n)
    for i in range(n):
        circuit.add_H_gate(i)
    for i in range(p):
        add_U_C(circuit, i)
        add_U_X(circuit, i)
    bound_circuit = circuit.bind_parameters(x)
    return quantum_state(n, circuit=bound_circuit)

cost_observable = Operator({pauli_label(f"Z{i} Z{(i + 1) % n}"): 0.5 for i in range(n)})
estimator = create_qulacs_vector_estimator()

def QAOA_cost_func(x: list[float], p: int) -> float:
    state = QAOA_state(x, p)
    return estimator(cost_observable, state).value.real
```

[図 5.6 プレースホルダ: QAOA で得られた頂点の分割法]

早速 $p=1$ での最適化を実行してみよう。

```python
p = 1
x0 = np.array([0.1] * (2 * p))
result = minimize(QAOA_cost_func, x0, args=(p), method='Powell')
print("QAOA Cost:", result.fun)
print("Optimized Parameter:", result.x)
```

```text
QAOA Cost: -0.9999999994991842
Optimized Parameter: [1.17809152 0.39269362]
```

得られた最適な状態 $\ket{\boldsymbol{\beta}^*,\boldsymbol{\gamma}^*}_{p=1}$ を測定したときにどのような値が得られるか見てみる。

```python
state_vec = evaluate_state_to_vector(QAOA_state(result.x, p)).vector
probs = np.abs(state_vec) ** 2
print(probs)

z_basis = [format(i, "b").zfill(n) for i in range(probs.size)]
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.xlabel("states")
plt.ylabel("probability (%)")
plt.bar(z_basis, probs * 100)
plt.show()
```

```text
[0.01562503 0.01562428 0.01562428 0.0781264  0.01562428 0.26562503
 0.0781264  0.01562428 0.01562428 0.0781264  0.26562503 0.01562428
 0.0781264  0.01562428 0.01562428 0.01562503]
```

[出力図プレースホルダ: p=1 の測定確率]

測定を行うと、`0101` か `1010` が測定される確率が高いことが分かる。これらのビット列は頂点 1 と頂点 3、頂点 2 と頂点 4 が同じグループになることを意味するから、長方形に対する最適な分割を表している。
この場合、図形を分割する曲線が横切る辺の数は 4 つであり、全部で辺は 4 つしかないから、この分割が最適解である。

つまり、最適化された $\ket{\boldsymbol{\beta}^*,\boldsymbol{\gamma}^*}_{p=1}$ に測定を行い、ある程度多数の測定結果を集めて測定確率が高いビット列を採用すれば、もともと解きたかった最大カット問題の解が得られたことになる。
一応、これでめでたしめでたしと言えるのだが、最適化されたコスト関数 $F(\boldsymbol{\beta},\boldsymbol{\gamma})$ の値が $-1$ だったことを思い出してほしい。
$C(Z)\ket{0101}=-2\ket{0101}$ であるから、コスト関数の真の最小値は $-2$ である。すなわち、コスト関数については正しい値が得られていないということである。
これは変分量子状態 $\ket{\boldsymbol{\beta},\boldsymbol{\gamma}}_{p=1}$ が十分な表現能力を持たず、コスト関数を最小化する真の解 $\ket{0101},\ket{1010}$ を表現できなかったことに由来すると考えられる。

そこで最後に、回路をより複雑にした $p=2$ の場合に結果がどう変わるか見てみよう。

```python
p = 2
x0 = np.array([0.1] * (2 * p))
result = minimize(QAOA_cost_func, x0, args=(p), method='Powell')
print("QAOA Cost:", result.fun)
print("Optimized parameters:", result.x)
state_vec = evaluate_state_to_vector(QAOA_state(result.x, p)).vector
probs = np.abs(state_vec) ** 2
print(probs)

plt.figure(figsize=(10, 5))
plt.xlabel("states")
plt.ylabel("probability (%)")
plt.bar(z_basis, probs * 100)
plt.show()
```

```text
QAOA Cost: -1.9999992381341494
Optimized parameters: [ 1.11854459  0.55939778 -2.12994593  0.4523998 ]
[1.52474693e-15 2.21896542e-08 2.21896542e-08 5.08539214e-08
 2.21896542e-08 4.99999810e-01 5.08539214e-08 2.21896542e-08
 2.21896542e-08 5.08539214e-08 4.99999810e-01 2.21896542e-08
 5.08539214e-08 2.21896542e-08 2.21896542e-08 1.52474693e-15]
```

[出力図プレースホルダ: p=2 の測定確率]

$p=1$ とは異なり、ほぼ確率 1 で真の解 $\ket{0101},\ket{1010}$ のどちらかが得られる状態になっている。
さらに、最適化後のコスト関数の値も真の最小値である $-2$ になっている。
このように、QAOA を用いる際には、変分量子回路の複雑さ $p$ の大きさにも注意しながら実装する必要がある。